# Phase 2: Exploratory Data Analysis (EDA)
## Project: Energy Consumption Prediction with MLOps
**Dataset**: Appliances Energy Prediction Dataset (UCI ML Repository)

### Objective:
Perform initial data exploration, analyze missing values, inspect target distribution (`Appliances`), investigate temporal patterns (hourly/daily trends), and assess correlations between indoor/outdoor weather features and energy usage.

In [ ]:
# 1. Setup and Imports
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data

# Graphics styling
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

--- 
### 2. Load Raw Dataset

In [ ]:
df = load_raw_data(config_path='../configs/config.yaml')
df.head()

--- 
### 3. Data Integrity & Schema Check

In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found!")

--- 
### 4. Target Variable Analysis (`Appliances`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Appliances'], kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribution of Appliances Energy Consumption (Wh)')
axes[0].set_xlabel('Appliances (Wh)')

sns.boxplot(x=df['Appliances'], ax=axes[1], color='coral')
axes[1].set_title('Boxplot of Appliances Energy Consumption')
axes[1].set_xlabel('Appliances (Wh)')

plt.tight_layout()
plt.show()

print("Summary Statistics of Target Variable:")
print(df['Appliances'].describe())

--- 
### 5. Time-Series Trends & Temporal Feature Extraction

In [ ]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()

plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x='hour', y='Appliances', estimator='mean', errorbar=None, marker='o', color='indigo')
plt.title('Average Energy Consumption by Hour of Day')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Average Appliances Energy Usage (Wh)')
plt.show()

--- 
### 6. Feature Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr[['Appliances']].sort_values(by='Appliances', ascending=False), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlations with Target (Appliances Energy Consumption)')
plt.show()

--- 
### 7. Insights for Phase 3 (Feature Engineering & Modeling)
1. **Target Skewness**: Target variable (`Appliances`) exhibits right skewness with occasional peak spikes. Log-transformation or robust scaling might improve linear models.
2. **Temporal Patterns**: Strong periodicity observed by hour of day (peak consumption during evening hours).
3. **Weather Correlations**: Temperature (`T2`, `T6`, `T_out`) and humidity (`RH_1`, `RH_out`) show significant relationship with energy consumption.
4. **Feature Candidates**: Lag features (e.g. energy 10 min ago, 1 hour ago), rolling statistics, cyclical time encodings (sin/cos of hour).